In [1]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

tr = pd.read_csv("../data/processed/split_train.csv")
te = pd.read_csv("../data/processed/split_test.csv")
for d in (tr, te):
    d["text"] = d["text"].fillna("")
    d["sub_category"] = d["sub_category"].fillna("unknown")

counts = tr["sub_category"].value_counts()
print("classes:", len(counts))
print(counts.head(5))
print(counts.tail(5))

classes: 100
sub_category
unknown                  2105
Games & Accessories      1653
Hobbies                  1538
Arts & Crafts            1424
Sports & Outdoor Play    1341
Name: count, dtype: int64
sub_category
Potty Training                         1
Fabric                                 1
Office Electronics                     1
Stationery & Gift Wrapping Supplies    1
PC                                     1
Name: count, dtype: int64


In [2]:
keep = counts[counts >= 30].index
tr_c = tr[tr["sub_category"].isin(keep)]
te_c = te[te["sub_category"].isin(keep)]
print(len(keep), "classes kept;", round(len(te_c) / len(te), 3), "of test rows remain")

41 classes kept; 0.976 of test rows remain


In [3]:
def score(name, model):
    model.fit(tr_c["text"], tr_c["sub_category"])
    pred = model.predict(te_c["text"])
    print(f"{name}: accuracy {accuracy_score(te_c['sub_category'], pred):.3f} | "
          f"macro F1 {f1_score(te_c['sub_category'], pred, average='macro'):.3f}")
    return pred

score("Most frequent class", DummyClassifier(strategy="most_frequent"))

svm = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)),
    ("clf", LinearSVC(class_weight="balanced")),
])
pred = score("TF-IDF + LinearSVC", svm)

Most frequent class: accuracy 0.087 | macro F1 0.004
TF-IDF + LinearSVC: accuracy 0.682 | macro F1 0.581


In [4]:
report = classification_report(te_c["sub_category"], pred, output_dict=True, zero_division=0)
r = pd.DataFrame(report).T.iloc[:-3].sort_values("f1-score")
r[["precision", "recall", "f1-score", "support"]].head(10)

,precision,recall,f1-score,support
Activity & Entertainment,0.000000,0.000000,0.000000,10.0
"Kids' Furniture, Décor & Storage",0.000000,0.000000,0.000000,13.0
Kitchen & Dining,0.037037,0.111111,0.055556,9.0
Men,0.055556,1.000000,0.105263,1.0
Office & School Supplies,0.129032,0.200000,0.156863,20.0
Fan Shop,0.400000,0.114286,0.177778,35.0
Remote & App Controlled Vehicles & Parts,0.125000,0.375000,0.187500,8.0
Event & Party Supplies,0.307692,0.266667,0.285714,30.0
unknown,0.377907,0.232143,0.287611,560.0
Dogs,0.307692,0.363636,0.333333,11.0
